Import dataset and libraries


In [ ]:
import pandas as pd
import tqdm
import numpy as np
import torch
from torch import nn
import os

In [ ]:
!unzip -q masked_sets.zip
!ls -l

total 657688
drwxr-xr-x 53 root root      4096 Oct 17  2012 rgbd-dataset_eval
-rw-r--r--  1 root root 673456874 Mar  7 03:53 rgbd-dataset_eval.zip
drwxr-xr-x  1 root root      4096 Feb  6 14:31 sample_data


Prepare data 



In [ ]:
from pathlib import Path
from sklearn.model_selection import train_test_split

def load_set(root, classes, class_to_idx):
    # Collect paired RGB and depth paths with labels
    X_rgb, X_depth, y = [], [], []

    # look for RGB files and depth files and separate into low, medium, and high occlusion
    for cls in classes:
        cls_dir = root / cls

        if not cls_dir.exists():
            continue

        rgb_paths = sorted([
            p for p in cls_dir.rglob("*_rgb.png")
            if "depthcrop" not in p.name and "maskcrop" not in p.name
        ])

        # For each RGB path, check if the corresponding depth path exists
        for rgb_path in rgb_paths:
            depth_path = Path(str(rgb_path).replace("_rgb_", "_depth_"))

            if depth_path.exists():
                X_rgb.append(str(rgb_path))
                X_depth.append(str(depth_path))
                y.append(class_to_idx[cls])
            else:
                print(f"Missing depth for {rgb_path}")

    return X_rgb, X_depth, y


# Path for training set
root_train = Path("masked_sets/train")

# Class folder for training set (master list)
classes = sorted([d.name for d in root_train.iterdir() if d.is_dir()])
class_to_idx = {c: i for i, c in enumerate(classes)}

# Load training set (paired)
X_rgb_all, X_depth_all, y_all = load_set(root_train, classes, class_to_idx)
print("Loaded training samples:", len(y_all))

# Split training set into train/val
X_rgb_train, X_rgb_val, X_depth_train, X_depth_val, y_train, y_val = train_test_split(
    X_rgb_all, X_depth_all, y_all,
    test_size=0.2,
    random_state=1234,
    stratify=y_all
)
print("Train samples:", len(y_train), "Val samples:", len(y_val))

# Paths for test sets
root_low  = Path("masked_sets/test/low_occlusion")
root_med  = Path("masked_sets/test/medium_occlusion")
root_high = Path("masked_sets/test/high_occlusion")

# Load test sets using the same classes & mapping
X_rgb_test_low,  X_depth_test_low,  y_test_low  = load_set(root_low,  classes, class_to_idx)
X_rgb_test_med,  X_depth_test_med,  y_test_med  = load_set(root_med,  classes, class_to_idx)
X_rgb_test_high, X_depth_test_high, y_test_high = load_set(root_high, classes, class_to_idx)

# Print test set sizes
print("Low occlusion test samples:", len(y_test_low))
print("Med occlusion test samples:", len(y_test_med))
print("High occlusion test samples:", len(y_test_high))

Create RGB and depth CNN model

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, in_channels, num_classes, feat_dim=256):
        super().__init__()
        # Simple CNN backbone
        self.backbone = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, padding=1), # 3x3 conv, 32 filters
            nn.ReLU(), 
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), # 3x3 conv, 64 filters
            nn.ReLU(), 
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), # 3x3 conv, 128 filters
            nn.ReLU(), 
            nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), # 3x3 conv, 256 filters
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)) # global average pooling to get 256-dim features
        )

    def forward(self, x):
        x = self.backbone(x)
        return x.view(x.size(0), -1) # flatten to (batch_size, feat_dim)

Create Fusion Model


In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

class RGBD_Fusion_Model(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        # RGB branch
        self.rgb_backbone = models.resnet18(weights=None)
        rgb_feature_dim = self.rgb_backbone.fc.in_features
        self.rgb_backbone.fc = nn.Identity()

        # Depth branch
        self.depth_backbone = models.resnet18(weights=None)
        depth_feature_dim = self.depth_backbone.fc.in_features
        self.depth_backbone.fc = nn.Identity()

        # if depth is 1-channel, change first conv layer
        self.depth_backbone.conv1 = nn.Conv2d(
            1, 64, kernel_size=7, stride=2, padding=3, bias=False
        )

        # fusion + classifier
        self.classifier = nn.Sequential(
            nn.Linear(rgb_feature_dim + depth_feature_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, rgb, depth):
        rgb_feat = self.rgb_backbone(rgb)
        depth_feat = self.depth_backbone(depth)

        fused_feat = torch.cat([rgb_feat, depth_feat], dim=1)
        out = self.classifier(fused_feat)
        return out

Train and test models

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def train(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0

    for rgb_imgs, depth_imgs, labels in tqdm.tqdm(dataloader):
        rgb_imgs = rgb_imgs.to(device)
        depth_imgs = depth_imgs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(rgb_imgs, depth_imgs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Total Loss = {total_loss / len(dataloader):.4f}")

def evaluate(models, dataloader, device):
    RGB_model, Depth_model = models
    RGB_model.eval()    
    Depth_model.eval()
    correct_rgb, correct_depth, total = 0, 0, 0
    with torch.no_grad():
        for rgb_imgs, depth_imgs, labels in dataloader:
            rgb_imgs, depth_imgs, labels = rgb_imgs.to(device), depth_imgs.to(device), labels.to(device)
            
            rgb_outputs = RGB_model(rgb_imgs)
            depth_outputs = Depth_model(depth_imgs)
            
            _, predicted_rgb = torch.max(rgb_outputs, 1)
            _, predicted_depth = torch.max(depth_outputs, 1)
            
            total += labels.size(0)
            correct_rgb += (predicted_rgb == labels).sum().item()
            correct_depth += (predicted_depth == labels).sum().item()
    
    print(f"RGB Accuracy: {correct_rgb / total * 100:.2f}%")
    print(f"Depth Accuracy: {correct_depth / total * 100:.2f}%")

In [ ]:
# Instantiate separate models for RGB and depth
cnn_rgb = SmallCNN(in_channels=3, num_classes=len(classes))
cnn_depth = SmallCNN(in_channels=1, num_classes=len(classes))
fusion_model = RGBD_Fusion_Model(num_classes=len(classes))

In [ ]:
from torch.utils.data import Dataset, DataLoader

# Create DataLoaders for training and testing
class RGBDepthDataset(Dataset):
    def __init__(self, rgb_paths, depth_paths, labels, transform=None):
        self.rgb_paths = rgb_paths
        self.depth_paths = depth_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        rgb_image = plt.imread(self.rgb_paths[idx]) / 255.0 # Normalize to [0,1]
        depth_image = plt.imread(self.depth_paths[idx]) / 255.0 # Normalize to [0,1]
        
        if self.transform:
            rgb_image = self.transform(rgb_image)
            depth_image = self.transform(depth_image)
        
        label = self.labels[idx]
        return rgb_image, depth_image, label

# Create Datasets
train_dataset = RGBDepthDataset(X_rgb_train, X_depth_train, y_train)
val_dataset = RGBDepthDataset(X_rgb_val, X_depth_val, y_val)
test_dataset_low = RGBDepthDataset(X_rgb_test_low, X_depth_test_low, y_test_low)
test_dataset_med = RGBDepthDataset(X_rgb_test_med, X_depth_test_med, y_test_med) 
test_dataset_high = RGBDepthDataset(X_rgb_test_high, X_depth_test_high, y_test_high)

# Create DataLoaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader_low = DataLoader(test_dataset_low, batch_size=batch_size, shuffle=False)
test_loader_med = DataLoader(test_dataset_med, batch_size=batch_size, shuffle=False)
test_loader_high = DataLoader(test_dataset_high, batch_size=batch_size, shuffle=False)

# Training loop
train_losses = []
test_accuracies = []
criterion = nn.CrossEntropyLoss()
optimizer_fusion = torch.optim.Adam(fusion_model.parameters(), lr=0.001)
epochs = 10
for epoch in range(epochs):
    print(f"Epoch {epoch+1}/{epochs}")
    train(fusion_model, train_loader, criterion, optimizer_fusion, device)
    print("Validation Results:")
    evaluate(fusion_model, val_loader, device)
    print("Low Occlusion Test Results:")
    evaluate(fusion_model, test_loader_low, device)
    print("Medium Occlusion Test Results:")
    evaluate(fusion_model, test_loader_med, device)
    print("High Occlusion Test Results:")
    evaluate(fusion_model, test_loader_high, device)


Combine the extraction features from both models using late fusion

Test the accuracy of the model using real scenes